# Differential Stable LatentMoE on BabyLM - Kaggle T4 x2

This notebook follows `diff-moe-3-scaled.ipynb`, but is dedicated to training the new **differential-attention + Kimi-style Stable LatentMoE** model from `feature/stable-latent-moe`. It is preconfigured for:

- config: `configs/s_diff_stable_latentmoe.yaml`
- run: `s_diff_stable_latentmoe`
- model: differential attention + Stable LatentMoE

Before running, set **Notebook Settings -> Accelerator -> GPU T4 x2**. Attach pre-tokenized tier-S BabyLM data when possible; preparing it inside a GPU session wastes quota. `RUN_FULL_TRAINING` is enabled, so **Run All** starts or resumes this differential latent-MoE run after setup and validation.


## 1. Clone the feature branch and install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

FEATURE_BRANCH = 'feature/stable-latent-moe'
REPO_URL = 'https://github.com/ramprasathk07/Differential-MOE.git'
REPO_DIR = Path('/kaggle/working/repo')

if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout; rename or remove it first.')
    subprocess.run([
        'git', 'clone', '--depth', '1', '--single-branch',
        '--branch', FEATURE_BRANCH, REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    current = subprocess.check_output(
        ['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True
    ).strip()
    if current != FEATURE_BRANCH:
        raise RuntimeError(f'Existing checkout is on {current!r}, expected {FEATURE_BRANCH!r}.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Using {FEATURE_BRANCH} at {commit} in {REPO_DIR}')


In [ ]:
# Fail early if the notebook is newer than the pushed feature branch.
import yaml
from src.model import Config
from src.model.moe import QuantileBalancedSigmoidGate, StableLatentMoE

STANDARD_REFERENCE_CONFIG = 'configs/s_stable_latentmoe.yaml'
DIFFERENTIAL_CONFIG = 'configs/s_diff_stable_latentmoe.yaml'
EXPECTED_ATTENTION = {
    STANDARD_REFERENCE_CONFIG: 'standard',
    DIFFERENTIAL_CONFIG: 'differential',
}
for path, attention in EXPECTED_ATTENTION.items():
    if not Path(path).is_file():
        raise RuntimeError(f'Missing {path}; push the latest feature branch and rerun setup.')
    cfg = Config.from_yaml(path)
    assert cfg.model.attention == attention
    assert cfg.model.ffn == 'stable_latent_moe'
    assert cfg.model.aux_loss_coef == 0.0 and cfg.model.router_z_coef == 0.0
print('Differential Stable LatentMoE code/config and its standard reference are available.')


In [ ]:
import torch

n_gpu = torch.cuda.device_count()
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', n_gpu)
for i in range(n_gpu):
    print(f'  cuda:{i}: {torch.cuda.get_device_name(i)}')
if n_gpu < 2:
    print('WARNING: select GPU T4 x2 for the intended DDP run; the notebook will fall back safely.')


## 2. Run settings

The training target is fixed to `s_diff_stable_latentmoe`. The standard-attention config is loaded only for a parity check and is never trained by this notebook.


In [ ]:
import json

USE_WANDB = False         # @param {type:'boolean'}
WANDB_PROJECT = 'diff-stable-latent-moe-kaggle'
MAX_STEPS_OVERRIDE = None  # e.g. 2400 after measuring the probe; None uses YAML
RUN_PROBE = False          # set True for a separate 100-step throughput probe
RUN_FULL_TRAINING = True
RUN_FINAL_TEST = False     # set True only after model selection is complete

CONFIG = DIFFERENTIAL_CONFIG
cfg = Config.from_yaml(CONFIG)
assert cfg.model.attention == 'differential'
assert cfg.model.ffn == 'stable_latent_moe'
run_name = cfg.run_name
TRACK = 'strict'
TOKENIZER = 'hf:Xenova/gpt-4'
FALLBACK_DATA_DIR = '/kaggle/working/data_s'
N_GPU = max(1, n_gpu)

print(f'config     : {CONFIG}')
print(f'run_name   : {run_name}')
print(f'attention  : {cfg.model.attention}')
print(f'ffn        : {cfg.model.ffn}')
print(f'latent/topk: {cfg.model.latent_dim}/{cfg.model.top_k}')
print(f'GPUs       : {N_GPU}')


In [ ]:
# Optional W&B authentication. Add WANDB_API_KEY under Kaggle Add-ons -> Secrets.
if USE_WANDB:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
        print('W&B key loaded.')
    except Exception as exc:
        USE_WANDB = False
        print('W&B disabled because WANDB_API_KEY is unavailable:', exc)


## 3. Resolve and verify tier-S data

Attach a Kaggle Dataset containing `train.bin`, `val.bin`, `test.bin`, `meta.json`, and `domain_offsets.json`. The resolver accepts extracted files or a ZIP. If no compatible attachment exists, the next cell prepares the strict BabyLM track with the cl100k-compatible tokenizer.


In [ ]:
import shutil
import zipfile

DATASET_PATH = None  # optional exact directory or ZIP under /kaggle/input
REQUIRED_DATA_FILES = ('train.bin', 'val.bin', 'test.bin', 'meta.json', 'domain_offsets.json')

def compatible_data_dir(path):
    path = Path(path)
    if not all((path / name).is_file() for name in REQUIRED_DATA_FILES):
        return False
    try:
        meta = json.loads((path / 'meta.json').read_text())
    except (OSError, ValueError):
        return False
    return meta.get('track') == TRACK and meta.get('tokenizer_spec') == TOKENIZER

def extract_token_zip(zip_path):
    destination = Path('/kaggle/working/attached_data')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        members = {Path(name).name: name for name in archive.namelist()}
        missing = [name for name in REQUIRED_DATA_FILES if name not in members]
        if missing:
            raise RuntimeError(f'{zip_path} is missing {missing}')
        for name in REQUIRED_DATA_FILES:
            with archive.open(members[name]) as src, open(destination / name, 'wb') as dst:
                shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    return destination

def resolve_data(explicit=None):
    if explicit:
        selected = Path(explicit)
        if not selected.exists():
            raise FileNotFoundError(selected)
        candidate = extract_token_zip(selected) if selected.is_file() else selected
        if not compatible_data_dir(candidate):
            raise RuntimeError(f'Incompatible tier-S data at {candidate}')
        return candidate
    root = Path('/kaggle/input')
    candidates = sorted({p.parent for p in root.rglob('meta.json')}) if root.exists() else []
    compatible = [p for p in candidates if compatible_data_dir(p)]
    if len(compatible) > 1:
        raise RuntimeError(f'Multiple compatible datasets found; set DATASET_PATH: {compatible}')
    if compatible:
        return compatible[0]
    if root.exists():
        for archive in sorted(root.rglob('*.zip')):
            try:
                extracted = extract_token_zip(archive)
            except (OSError, zipfile.BadZipFile, RuntimeError):
                continue
            if compatible_data_dir(extracted):
                return extracted
    return None

resolved = resolve_data(DATASET_PATH)
DATA_DIR = str(resolved) if resolved else FALLBACK_DATA_DIR
USING_ATTACHED = resolved is not None
print('Using data:', DATA_DIR)
print('Attached/pre-tokenized:', USING_ATTACHED)


In [ ]:
data_dir_path = Path(DATA_DIR)
if not (data_dir_path / 'train.bin').is_file():
    print('No attached token data found; preparing strict BabyLM data now.')
    subprocess.run([
        sys.executable, '-m', 'src.data.prepare',
        '--tokenizer', TOKENIZER, '--out_dir', DATA_DIR, '--track', TRACK,
    ], check=True)
else:
    print('Tokenized data already present; preparation skipped.')

subprocess.run([
    sys.executable, '-m', 'src.data.verify',
    '--data_dir', DATA_DIR, '--config', CONFIG,
], check=True)


## 4. Confirm the differential training target and reference parity

In [ ]:
from dataclasses import asdict

standard_cfg = Config.from_yaml(STANDARD_REFERENCE_CONFIG)
differential_cfg = Config.from_yaml(DIFFERENTIAL_CONFIG)
standard_model = asdict(standard_cfg.model)
differential_model = asdict(differential_cfg.model)
standard_model.pop('attention')
differential_model.pop('attention')
assert standard_model == differential_model, 'The pair differs in fields other than attention.'
assert asdict(standard_cfg.train) == asdict(differential_cfg.train)
assert differential_cfg.run_name == 's_diff_stable_latentmoe'
print('Training target verified: differential attention + Stable LatentMoE.')
print('Reference parity verified: only the attention type and run_name differ.')
subprocess.run([
    sys.executable, '-m', 'src.params', '--config',
    STANDARD_REFERENCE_CONFIG, DIFFERENTIAL_CONFIG,
], check=True)


## 5. Optional throughput probe

Set `RUN_PROBE = True` in the settings cell to run 100 steps in a separate output directory. This avoids contaminating the full run's checkpoint and cosine schedule.


In [ ]:
def train_command(out_dir, max_steps=None):
    if N_GPU > 1:
        cmd = [
            sys.executable, '-m', 'torch.distributed.run', '--standalone',
            f'--nproc_per_node={N_GPU}', '-m', 'src.train',
        ]
    else:
        cmd = [sys.executable, '-m', 'src.train']
    cmd += ['--config', CONFIG, '--data_dir', DATA_DIR, '--out_dir', out_dir]
    if N_GPU > 1:
        cmd.append('--ddp')
    if USE_WANDB:
        cmd += ['--wandb', '--wandb_project', WANDB_PROJECT]
    if max_steps is not None:
        cmd += ['--max_steps', str(max_steps)]
    return cmd

def run_training(out_dir, max_steps=None):
    env = os.environ.copy()
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    cmd = train_command(out_dir, max_steps)
    print('Launching:', ' '.join(cmd))
    subprocess.run(cmd, check=True, env=env)

if RUN_PROBE:
    run_training('/kaggle/working/probe', max_steps=100)
else:
    print('Probe skipped. Set RUN_PROBE = True in the settings cell to enable it.')


In [ ]:
import pandas as pd

HOURS_PER_RUN = 7.5
probe_csv = Path('/kaggle/working/probe') / run_name / 'metrics.csv'
if probe_csv.is_file():
    probe_df = pd.read_csv(probe_csv)
    rates = probe_df['tok_per_sec'].dropna()
    measured = rates.iloc[1:].median() if len(rates) > 1 else rates.iloc[0]
    tokens_per_step = cfg.train.batch_size * cfg.model.seq_len * cfg.train.accum_steps * N_GPU
    affordable = int(measured * HOURS_PER_RUN * 3600 / tokens_per_step)
    print(f'Measured: {measured:,.0f} tok/s; {tokens_per_step:,} tokens/step')
    print(f'Affordable in {HOURS_PER_RUN} h: {affordable:,} steps')
    print(f'Set MAX_STEPS_OVERRIDE = {affordable} in the settings cell if desired.')
else:
    print('No probe metrics yet.')


## 6. Full training run

Re-running this cell resumes automatically from `/kaggle/working/checkpoints/<run_name>/last.pt`. Quantile-balancing bias updates are committed once after each optimizer step and stored in checkpoints.


In [ ]:
if RUN_FULL_TRAINING:
    run_training('/kaggle/working/checkpoints', max_steps=MAX_STEPS_OVERRIDE)
else:
    print('Full training skipped by RUN_FULL_TRAINING = False.')


## 7. Inspect training results and routing diagnostics

In [ ]:
import matplotlib.pyplot as plt

RUN_DIR = Path('/kaggle/working/checkpoints') / run_name
report_path = RUN_DIR / 'report.json'
metrics_path = RUN_DIR / 'metrics.csv'
if report_path.is_file():
    print(json.dumps(json.loads(report_path.read_text()), indent=2))
else:
    print('No report yet:', report_path)

if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    metrics.dropna(subset=['train_loss']).plot(
        x='step', y='train_loss', ax=axes[0], title=f'{run_name}: train loss'
    )
    metrics.dropna(subset=['val_nll']).plot(
        x='step', y='val_nll', ax=axes[1], marker='o', title=f'{run_name}: val NLL'
    )
    plt.tight_layout()
    plt.show()
    display(metrics.tail(10))
else:
    print('No metrics yet:', metrics_path)


## 8. One-time held-out test evaluation

Keep `RUN_FINAL_TEST = False` during tuning. Enable it only after the best checkpoint has been selected using validation NLL. The result includes expert entropy/imbalance, quantile router biases, and differential-attention lambda diagnostics.


In [ ]:
if RUN_FINAL_TEST:
    subprocess.run([
        sys.executable, '-m', 'src.eval',
        '--config', CONFIG, '--run_dir', str(RUN_DIR),
        '--data_dir', DATA_DIR, '--device', 'cuda' if torch.cuda.is_available() else 'cpu',
    ], check=True)
    final_report = json.loads((RUN_DIR / 'final_test_eval.json').read_text())
    print(json.dumps(final_report, indent=2))
else:
    print('Held-out test evaluation skipped. Enable RUN_FINAL_TEST only after model selection.')


## 9. Persist checkpoints across sessions

Kaggle wipes `/kaggle/working` between sessions. Use **Save Version** to preserve outputs. To resume later, attach the previous output and copy its `checkpoints` directory back into the writable `/kaggle/working/checkpoints` path before running training.


In [ ]:
if RUN_DIR.is_dir():
    for path in sorted(RUN_DIR.rglob('*')):
        if path.is_file():
            print(path.relative_to(RUN_DIR), f'{path.stat().st_size / 1e6:.1f} MB')
else:
    print('No checkpoint directory yet:', RUN_DIR)

# Example for a future session (edit the attached dataset path):
# !mkdir -p /kaggle/working/checkpoints
# !cp -r /kaggle/input/<previous-output>/checkpoints/* /kaggle/working/checkpoints/
